In [0]:
import pyspark.sql.functions as F
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("catalog_name", "ecommerce", "Catalog Name")
dbutils.widgets.text("storage_account_name", "ecommcistorage05", "Storage Account Name")
dbutils.widgets.text("container_name", "ecomm-raw-data", "Container Name")

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")
storage_account_name = dbutils.widgets.get("storage_account_name")
container_name = dbutils.widgets.get("container_name")

In [0]:
df_batch = spark.read.table(f"{catalog_name}.bronze.brz_shipments")
display(df_batch.limit(20))

order_dt,shipment_id,order_id,carrier,_rescued_data,ingest_timestamp,source_file
2025-08-01,S643611,643611,BlueDart,null,2026-09-27T13:37:53.997Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_shipments/landing/shipments_2025-08.csv
2025-08-01,S643612,643612,AusPost,null,2026-09-27T13:37:53.997Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_shipments/landing/shipments_2025-08.csv
2025-08-01,S643613,643613,DHL,null,2026-09-27T13:37:53.997Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_shipments/landing/shipments_2025-08.csv
2025-08-01,S643614,643614,XpressBees,null,2026-09-27T13:37:53.997Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_shipments/landing/shipments_2025-08.csv
2025-08-01,S643615,643615,Delhivery,null,2026-09-27T13:37:53.997Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_shipments/landing/shipments_2025-08.csv
2025-08-01,S643616,643616,BlueDart,null,2026-09-27T13:37:53.997Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_shipments/landing/shipments_2025-08.csv
2025-08-01,S643617,643617,DHL,null,2026-09-27T13:37:53.997Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_shipments/landing/shipments_2025-08.csv
2025-08-01,S643618,643618,DHL,null,2026-09-27T13:37:53.997Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_shipments/landing/shipments_2025-08.csv
2025-08-01,S643619,643619,DHL,null,2026-09-27T13:37:53.997Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_shipments/landing/shipments_2025-08.csv
2025-08-01,S643620,643620,DHL,null,2026-09-27T13:37:53.997Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_shipments/landing/shipments_2025-08.csv


In [0]:
df = spark.readStream.format("delta").table(f"{catalog_name}.bronze.brz_shipments")

### Convert `order_dt` to DateType

In [0]:
df = df.withColumn("order_dt",F.to_date(F.col("order_dt")))

### Convert `carrier` column to uppercase and trim whitespace

In [0]:
df = df.withColumn("carrier",F.upper(F.trim(F.col("carrier"))))

### Add `processed_time` column with the current timestamp.

In [0]:
df = df.withColumn("processed_time",F.current_timestamp())

### Saving the table in silver layer

In [0]:
# Silver checkpoint
silver_checkpoint_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoint/silver/fact_shipments/"
print(silver_checkpoint_path)

abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/checkpoint/silver/fact_shipments/


In [0]:
df_batch.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_batch.columns
]).show()


+--------+-----------+--------+-------+-------------+----------------+-----------+
|order_dt|shipment_id|order_id|carrier|_rescued_data|ingest_timestamp|source_file|
+--------+-----------+--------+-------+-------------+----------------+-----------+
|       0|          0|       0|      0|       679731|               0|          0|
+--------+-----------+--------+-------+-------------+----------------+-----------+



In [0]:
# checking which column can be considered as primary key.
def check_key(df, cols):
    total = df.count()
    distinct = df.select(*cols).distinct().count()
    nulls = df.filter(" OR ".join([f"{c} IS NULL" for c in cols])).count()
    print(f"{cols} -> total={total}, distinct={distinct}, nulls={nulls}, "
          f"is_unique={total == distinct}, no_nulls={nulls == 0}")

In [0]:
check_key(df_batch, ["order_id"])
check_key(df_batch, ["order_id", "shipment_id"])

# see the offenders if orderid isn't unique
df_batch.groupBy("order_id").count().filter("count > 1").show()

['order_id'] -> total=679731, distinct=679731, nulls=0, is_unique=True, no_nulls=True
['order_id', 'shipment_id'] -> total=679731, distinct=679731, nulls=0, is_unique=True, no_nulls=True
+--------+-----+
|order_id|count|
+--------+-----+
+--------+-----+



In [0]:
def upsert_to_silver(microBatchDF, batchId):
    table_name = f"{catalog_name}.silver.slv_shipments"
    if not spark.catalog.tableExists(table_name):
        print("creating new table")
        microBatchDF.write.format("delta").mode("overwrite").saveAsTable(table_name)
        spark.sql(
            f"ALTER TABLE {table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
        )
    else:
        deltaTable = DeltaTable.forName(spark, table_name)
        deltaTable.alias("silver_table").merge(
            microBatchDF.alias("batch_table"),
            "silver_table.order_id = batch_table.order_id" and "silver_table.shipment_id = batch_table.shipment_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()  

In [0]:
df.writeStream.trigger(availableNow=True).foreachBatch(
    upsert_to_silver
).format("delta").option("checkpointLocation", silver_checkpoint_path).option(
    "mergeSchema", "true"
).outputMode(
    "update"
).trigger(
    once=True
).start().awaitTermination()